Reddit Sentiment Analysis Project

Collecting comments from r/ChatGPT, r/ClaudeAI, and r/Gemini to analyze public sentiment towards AI models.

Data Cleaning and Preparation



Importing Libraries

In [82]:
#import libraries
import pandas as pd
import html
import langid
from concurrent.futures import ThreadPoolExecutor

In [83]:
#read the file

rdf = pd.read_csv('C:/Users/sediq/Downloads/reddit_data_raw 1.csv', low_memory=False)

In [84]:
# Check the data
print(rdf.shape)
print(rdf.head())

(1403180, 8)
  subreddit  post_id post_title     type  \
0   ChatGPT  1ncz57y        NaN  comment   
1   ChatGPT  1ne13tx        NaN  comment   
2   ChatGPT  1nebx17        NaN  comment   
3   ChatGPT  1ndtzc8        NaN  comment   
4   ChatGPT  1nejs7v        NaN  comment   

                                                text  score  upvote_ratio  \
0  https://preview.redd.it/1lh448otdmof1.png?widt...      2           NaN   
1  You replace one neuron with a machine like cou...      1           NaN   
2  that's an eye opening (not open a eyeing) pers...      2           NaN   
3  https://preview.redd.it/tlr7i3b0emof1.png?widt...      3           NaN   
4  The bot should be public domain with public co...      2           NaN   

         date  
0  2025-09-11  
1  2025-09-11  
2  2025-09-11  
3  2025-09-11  
4  2025-09-11  


Step 0: Check the baseline

In [85]:
print(f"Total rows: {len(rdf)}")
print(f"Total columns: {len(rdf.columns)}")
print(f"\nColumn names: {rdf.columns.tolist()}")

Total rows: 1403180
Total columns: 8

Column names: ['subreddit', 'post_id', 'post_title', 'type', 'text', 'score', 'upvote_ratio', 'date']


Step 1: Appropriate format of all the columns

In [86]:
# post_id — check pattern (5-7 char lowercase alphanumeric)
print("\n--- post_id ---")
invalid_post_ids = rdf[~rdf['post_id'].str.match(r'^[a-z0-9]{5,7}$', na=False)]
print(f"Invalid format post_ids: {len(invalid_post_ids):,}")

# subreddit — should only have 3 values
print("\n--- subreddit ---")
print(f"Unique values: {rdf['subreddit'].unique().tolist()}")

# type — should only be post or comment
print("\n--- type ---")
print(f"Unique values: {rdf['type'].unique().tolist()}")

# score — should be numeric integer
print("\n--- score ---")
print(f"Dtype: {rdf['score'].dtype}")

# upvote_ratio — should be numeric
print("\n--- upvote_ratio ---")
print(f"Dtype: {rdf['upvote_ratio'].dtype}")

# Date format checker
print("\n--- Date ---")
print(rdf['date'].dtype)

# fill the row with NaN when the conversion raises an error
rdf['date'] = pd.to_datetime(rdf['date'], errors='coerce')


--- post_id ---
Invalid format post_ids: 0

--- subreddit ---
Unique values: ['ChatGPT', 'ClaudeAI', 'Gemini']

--- type ---
Unique values: ['comment', 'post']

--- score ---
Dtype: int64

--- upvote_ratio ---
Dtype: float64

--- Date ---
str


Step 2: Removing the links from the text

In [87]:
rdf['text'] = rdf['text'].str.replace(r'http\S+', '', regex=True)
rdf['text'] = rdf['text'].str.replace(r'www\S+', '', regex=True)

# Drop rows that were ONLY a URL (now empty after removal)
rdf = rdf[rdf['text'].str.strip() != '']
print(f"After removing URLs: {len(rdf):,}")

After removing URLs: 1,371,806


Step 3: Remove the rows with text less than 10 characters, as that doesn't give meaningful sentiments.

In [88]:
print("\nUnique short texts and their number of occurance:")
short_texts = rdf[rdf['text'].str.len() < 10]['text'].value_counts().head(20)
print(short_texts)

rdf = rdf[rdf['text'].str.len() >= 10]
print(f"After removing short comments (<10 chars): {len(rdf):,}")


Unique short texts and their number of occurance:
text
[            1401
Yes           561
😂             494
Same          453
lol           409
Lol           350
No            342
🤣             287
Lmao          248
Thanks!       241
😂😂😂           199
Thanks        199
Yes.          186
🤣🤣🤣           186
Why?          177
No.           171
Thank you     167
What?         162
same          158
LOL           155
Name: count, dtype: int64
After removing short comments (<10 chars): 1,263,796


Step 4: Removing the duplicates

In [89]:
rdf_clean = rdf.drop_duplicates(subset=['post_id', 'text'], keep='first')
print(f"After removing duplicates: {len(rdf_clean):,}")


After removing duplicates: 1,263,325


Step 5: Removing the Null and Empty values

In [90]:
rdf_clean = rdf_clean.dropna(subset=['text', 'date'])
rdf_clean = rdf_clean[rdf_clean['text'].str.strip() != '']
print(f"After removing nulls/empty: {len(rdf_clean):,}")

After removing nulls/empty: 1,263,325


Step 6: Removing the deleted posts and comments

In [91]:
text_stripped = rdf_clean['text'].fillna('').str.strip().str.lower()
remove_mask = text_stripped.isin(['[removed]', 'removed', '[removed by reddit]',
                                   '[deleted]', 'deleted', '[deleted by user]'])
rdf_clean = rdf_clean[~remove_mask]
print(f"After removing [removed]/[deleted]: {len(rdf_clean):,}")


After removing [removed]/[deleted]: 1,263,325


Step 7: Decoding HTML entities, user mentions and removing Reddit markdown.

In [92]:
# Decode HTML entities
rdf_clean['text'] = rdf_clean['text'].apply(html.unescape)

# Remove Reddit markdown formatting
rdf_clean['text'] = rdf_clean['text'].str.replace(r'\*\*|\*|~~|__', '', regex=True)

# Remove subreddit and user mentions
rdf_clean['text'] = rdf_clean['text'].str.replace(r'r/\w+', '', regex=True)
rdf_clean['text'] = rdf_clean['text'].str.replace(r'u/\w+', '', regex=True)

print(f"After HTML and special character cleaning: {len(rdf_clean):,}")

After HTML and special character cleaning: 1,263,325


Step 8: Leaving only the English comments

In [93]:
def detect_lang(text):
    try:
        lang, _ = langid.classify(text)
        return lang
    except:
        return 'unknown'

# Run in parallel, faster than row by row
with ThreadPoolExecutor(max_workers=4) as executor:
    rdf_clean['language'] = list(executor.map(detect_lang, rdf_clean['text']))

print(rdf_clean['language'].value_counts())

# Keep only English
rdf_clean = rdf_clean[rdf_clean['language'] == 'en']
print(f"After keeping only English: {len(rdf_clean):,}")

KeyboardInterrupt: 

Step 9: Writing the DataFrame to a CSV

In [ ]:
rdf_clean.to_csv('reddit_data_cleaned.csv', index=False)

print(f"Saved {len(rdf_clean):,} rows to reddit_data_cleaned.csv")